In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

In [2]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from tremelique._acoustic import Acoustic
from tremelique._wavelets import RickerWavelet
from tremelique._utils import anim_to_html, model

In [3]:
shape = (300, 1000) # (nz, nx), notebook do Leo
region = [0, 10000, 0, 3000]
spacing = 10.0      # (dx, dz)
interface_idx = 100 #notebook do Leo
interface_depth = 1000 # interface_idx * spacing = profundidade em metros

camadas = [
    # Camada 1: 0 a 1000m
    {"z": 0, "velocity": 4000, "density": 2000},
    # Camada 2: 1000m para baixo
    {"z": interface_depth, "velocity": 5000, "density": 2500}
]

model = model(region=region, spacing=spacing, layers=camadas)

In [4]:
shot = Acoustic(model, taper=0.004, padding=60)

fonte = (0, 100) # (nz, nx)
shot.add_point_source(fonte, RickerWavelet(amp=1, f_cut=60)) #agora é possivel fazer add_point_source_physical (1000, 0) -> (x, z)
shot.run(3000)

Output()

C:\Users\paulo\miniforge3\envs\tremelique\Lib\site-packages\numba\core\typed_passes.py:336: 
NumbaPerformanceWarning: 
The keyword argument 'parallel=True' was specified but no transformation for parallel execution was possible.

To find out why, try turning on parallel diagnostics, see 
https://numba.readthedocs.io/en/stable/user/parallel.html#diagnostics for help.

File "..\tremelique\_acoustic.py", line 533:
@numba.jit(nopython=True, parallel=True)
def timestep_esg(u_tp1, u_t, u_tm1, x1, x2, z1, z2, dt, dx, dz, vel, dens):
^

  warnings.warn(errors.NumbaPerformanceWarning(msg,

In [5]:
#Processamento de dados
every = 20
frames = shot.simsize // every

#Receptores
receptores = np.arange(fonte[1] + 50, shape[1], 15, dtype='int')
x = receptores * spacing

#Acesso o dataset via .wavefield
ds = shot.wavefield 
#Slice: [time, z=0, x=receptores]
dados = ds["panels"][:, 0, receptores].values 
times = ds["time"].values

dados_dummy = np.zeros_like(dados)

#Plotagem
fig, axes = plt.subplots(2, 1, sharex=True, sharey=False, figsize=(12, 8), facecolor='white')

#Sismograma
ax1 = axes[0]
ax1.set_title('iteration: 0')
ax1.set_ylabel('time (s)')

cutoff = 0.4 
section = ax1.imshow(dados_dummy, extent=[x.min(), x.max(), times.max(), times.min()], 
                     aspect='auto', cmap='Greys', vmin=-cutoff, vmax=cutoff,
                     interpolation='nearest')
ax1.set_ylim(0, times.max())

#Campo de Onda
ax2 = axes[1]
ax2.set_xlabel('x (m)')
ax2.set_ylabel('depth (m)')

cutoff = 1 
extent = [0, shape[1]*spacing, shape[0]*spacing, 0] # [xmin, xmax, zmax, zmin]

wavefield = ax2.imshow(shot[0], extent=extent, vmin=-cutoff, vmax=cutoff, cmap='Greys')

#Marcadores
ax2.plot(fonte[1]*spacing, 0, '*y', markersize=15)
ax2.plot(x, np.zeros_like(receptores), 'vb', markersize=10)
ax2.hlines(interface_depth, 0, shape[1]*spacing, color='k')
fig.tight_layout(h_pad=0)

#Animação
def anim_shot(frame):
    idx = frame * every
    ax1.set_title('iteration: {:d}'.format(idx))
    
    u = shot[idx] 
    wavefield.set_array(u)
    
    dados_dummy[:idx, :] = dados[:idx, :]
    section.set_array(dados_dummy)
    return wavefield, section

#Gera o video
anim = FuncAnimation(fig, anim_shot, frames=frames)
anim_to_html(anim, fps=6, dpi=60)